In [1]:
import numpy as np
import imageio
import os


def add_noise(image, sigma=50):
    image = np.array(image / 255, dtype=float)
    noise = np.random.normal(0, sigma / 255, image.shape)
    gauss_noise = image + noise
    return gauss_noise * 255


def save_image(image, path):
    image = np.round(np.clip(image, 0, 255)).astype(np.uint8)
    imageio.imwrite(path, image)


def crop_image(image, s=8):
    h, w, c = image.shape
    return image[:h - h % s, :w - w % s, :]


clean_dir = "/kaggle/input/div-2k-train-hr"
noisy_dir = "/kaggle/working/DIV2K_noisy_sigma50"

os.makedirs(noisy_dir, exist_ok=True)

for image_name in sorted(os.listdir(clean_dir)):
    if not image_name.endswith(".png"):
        continue

    img_path = os.path.join(clean_dir, image_name)
    img = imageio.imread(img_path)

    img = crop_image(img)
    img_noise = add_noise(img, sigma=50)

    save_image(img_noise, os.path.join(noisy_dir, image_name))

print("✅ Noisy images created")


/tmp/ipykernel_55/3936354704.py:33: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(img_path)


✅ Noisy images created


In [10]:
import torch
from torch.utils.data import Dataset
import imageio
import os
import torchvision.transforms as T


class DenoiseDataset(Dataset):

    def __init__(self, noisy_dir, clean_dir):

        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir

        self.files = sorted(os.listdir(noisy_dir))

        self.to_tensor = T.ToTensor()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        name = self.files[idx]

        noisy = imageio.imread(os.path.join(self.noisy_dir, name))
        clean = imageio.imread(os.path.join(self.clean_dir, name))

        noisy = self.to_tensor(noisy)
        clean = self.to_tensor(clean)

        return noisy, clean

In [3]:
!git clone https://github.com/swz30/Restormer.git

Cloning into 'Restormer'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 312 (delta 74), reused 72 (delta 72), pack-reused 197 (from 2)
Receiving objects: 100% (312/312), 1.55 MiB | 10.09 MiB/s, done.
Resolving deltas: 100% (131/131), done.


In [19]:
!mkdir -p Restormer/pretrained_models

!wget https://github.com/swz30/Restormer/releases/download/v1.0/gaussian_color_denoising_sigma50.pth \
-O Restormer/pretrained_models/gaussian_color_denoising_sigma50.pth

--2026-03-16 03:34:17--  https://github.com/swz30/Restormer/releases/download/v1.0/gaussian_color_denoising_sigma50.pth
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/418793252/db623618-23e9-450e-86de-2ff6ebbf2202?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-16T04%3A32%3A44Z&rscd=attachment%3B+filename%3Dgaussian_color_denoising_sigma50.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-16T03%3A31%3A58Z&ske=2026-03-16T04%3A32%3A44Z&sks=b&skv=2018-11-09&sig=7IBIAhi77XNp19%2Bq3nOc0YOjDG9gjLCt4z5d2VLlyJA%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3MzYzMzg1NywibmJmIjoxNzczNjMyMDU3LCJwYXRo

In [8]:
!pip install einops lmdb natsort yacs tqdm opencv-python scikit-image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 6.4 MB/s eta 0:00:0000:01


In [18]:
import os

for root, dirs, files in os.walk("/kaggle"):
    for file in files:
        if "sigma50" in file:
            print(os.path.join(root, file))

/kaggle/working/restormer_sigma50_best.pth


In [22]:
# ==============================
# Restormer Denoising Training
# ==============================

import os
import numpy as np
import imageio
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
import sys


print("Starting Restormer Training Pipeline...")


# ==============================
# 1. Clone Restormer
# ==============================

if not os.path.exists("Restormer"):
    print("Cloning Restormer repository...")
    os.system("git clone https://github.com/swz30/Restormer.git")

sys.path.append("Restormer")

from basicsr.models.archs.restormer_arch import Restormer

print("Restormer imported successfully")


# ==============================
# 2. Paths
# ==============================

clean_dir = "/kaggle/input/div-2k-train-hr"
noisy_dir = "/kaggle/working/DIV2K_noisy_sigma50"

pretrained_path = "Restormer/pretrained_models/gaussian_color_denoising_sigma50.pth"

print("Clean dataset:", clean_dir)
print("Noisy dataset:", noisy_dir)


# ==============================
# 3. Dataset Loader
# ==============================

class DenoiseDataset(Dataset):

    def __init__(self, noisy_dir, clean_dir, patch_size=128):

        self.noisy_dir = noisy_dir
        self.clean_dir = clean_dir
        self.files = sorted(os.listdir(noisy_dir))

        self.patch_size = patch_size
        self.to_tensor = T.ToTensor()

        print("Dataset initialized with", len(self.files), "images")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        name = self.files[idx]

        noisy = imageio.imread(os.path.join(self.noisy_dir, name))
        clean = imageio.imread(os.path.join(self.clean_dir, name))

        h, w, _ = noisy.shape

        x = np.random.randint(0, h - self.patch_size)
        y = np.random.randint(0, w - self.patch_size)

        noisy = noisy[x:x+self.patch_size, y:y+self.patch_size]
        clean = clean[x:x+self.patch_size, y:y+self.patch_size]

        noisy = self.to_tensor(noisy)
        clean = self.to_tensor(clean)

        return noisy, clean


# ==============================
# 4. Dataset + Loader
# ==============================

print("Loading dataset...")

dataset = DenoiseDataset(noisy_dir, clean_dir, patch_size=128)

train_loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2
)

print("Dataset size:", len(dataset))


# ==============================
# 5. Load Restormer Model
# ==============================

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

model = Restormer(
    inp_channels=3,
    out_channels=3,
    dim=48,
    num_blocks=[4,6,6,8],
    num_refinement_blocks=4,
    heads=[1,2,4,8],
    ffn_expansion_factor=2.66,
    bias=False,
    LayerNorm_type='BiasFree'
)

model = model.to(device)

print("Restormer model created")


# ==============================
# 6. Load Pretrained Weights (FIXED)
# ==============================

if os.path.exists(pretrained_path):

    print("Loading pretrained weights...")

    checkpoint = torch.load(pretrained_path, map_location=device)

    model.load_state_dict(checkpoint["params"])   # <-- FIX HERE

    print("Pretrained Restormer weights loaded successfully")

else:

    print("WARNING: pretrained weights not found")


# ==============================
# 7. Loss + Optimizer
# ==============================

criterion = nn.L1Loss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("Optimizer initialized")


# ==============================
# 8. PSNR Function
# ==============================

def psnr(img1, img2):

    mse = torch.mean((img1 - img2) ** 2, dim=[1,2,3])
    psnr = 10 * torch.log10(1 / mse)

    return torch.mean(psnr)


# ==============================
# 9. Training Loop
# ==============================

epochs = 50
best_loss = 999

print("Starting training...")

for epoch in range(epochs):

    print("\n==============================")
    print(f"Epoch {epoch+1}/{epochs}")
    print("==============================")

    model.train()

    total_loss = 0
    total_psnr = 0

    for i, (noisy, clean) in enumerate(train_loader):

        noisy = noisy.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()

        output = model(noisy)

        loss = criterion(output, clean)

        loss.backward()

        optimizer.step()

        batch_psnr = psnr(output.detach(), clean)

        total_loss += loss.item()
        total_psnr += batch_psnr.item()

        if i % 20 == 0:

            print(
                f"Batch {i}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"PSNR: {batch_psnr:.2f}"
            )

    avg_loss = total_loss / len(train_loader)
    avg_psnr = total_psnr / len(train_loader)

    print("\nEpoch Summary")
    print("Average Loss:", avg_loss)
    print("Average PSNR:", avg_psnr)

    if avg_loss < best_loss:

        best_loss = avg_loss

        torch.save(model.state_dict(), "restormer_sigma50_best.pth")

        print("New best model saved")


print("\nTraining complete")

Starting Restormer Training Pipeline...
Restormer imported successfully
Clean dataset: /kaggle/input/div-2k-train-hr
Noisy dataset: /kaggle/working/DIV2K_noisy_sigma50
Loading dataset...
Dataset initialized with 800 images
Dataset size: 800
Using device: cuda
Restormer model created
Loading pretrained weights...
Pretrained Restormer weights loaded successfully
Optimizer initialized
Starting training...

Epoch 1/50


/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:72: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  clean = imageio.imrea

Batch 0/400 | Loss: 0.0359 | PSNR: 27.86
Batch 20/400 | Loss: 0.0367 | PSNR: 27.26
Batch 40/400 | Loss: 0.0349 | PSNR: 27.66
Batch 60/400 | Loss: 0.0309 | PSNR: 28.11
Batch 80/400 | Loss: 0.0312 | PSNR: 28.34
Batch 100/400 | Loss: 0.0343 | PSNR: 27.06
Batch 120/400 | Loss: 0.0304 | PSNR: 28.12
Batch 140/400 | Loss: 0.0228 | PSNR: 31.09
Batch 160/400 | Loss: 0.0486 | PSNR: 24.42
Batch 180/400 | Loss: 0.0235 | PSNR: 30.49
Batch 200/400 | Loss: 0.0252 | PSNR: 28.89
Batch 220/400 | Loss: 0.0474 | PSNR: 24.60
Batch 240/400 | Loss: 0.0483 | PSNR: 25.30
Batch 260/400 | Loss: 0.0180 | PSNR: 33.49
Batch 280/400 | Loss: 0.0246 | PSNR: 30.56
Batch 300/400 | Loss: 0.0277 | PSNR: 28.54
Batch 320/400 | Loss: 0.0179 | PSNR: 34.68
Batch 340/400 | Loss: 0.0338 | PSNR: 26.97
Batch 360/400 | Loss: 0.0189 | PSNR: 33.18
Batch 380/400 | Loss: 0.0264 | PSNR: 29.07

Epoch Summary
Average Loss: 0.029860140094533562
Average PSNR: 29.17151243209839
New best model saved

Epoch 2/50


/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:72: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  clean = imageio.imrea

Batch 0/400 | Loss: 0.0263 | PSNR: 30.18
Batch 20/400 | Loss: 0.0194 | PSNR: 31.73
Batch 40/400 | Loss: 0.0354 | PSNR: 27.85
Batch 60/400 | Loss: 0.0171 | PSNR: 32.46
Batch 80/400 | Loss: 0.0335 | PSNR: 27.37
Batch 100/400 | Loss: 0.0339 | PSNR: 27.11
Batch 120/400 | Loss: 0.0291 | PSNR: 28.25
Batch 140/400 | Loss: 0.0185 | PSNR: 31.64
Batch 160/400 | Loss: 0.0249 | PSNR: 29.91
Batch 180/400 | Loss: 0.0201 | PSNR: 31.92
Batch 200/400 | Loss: 0.0384 | PSNR: 25.83
Batch 220/400 | Loss: 0.0288 | PSNR: 31.73
Batch 240/400 | Loss: 0.0198 | PSNR: 31.58
Batch 260/400 | Loss: 0.0266 | PSNR: 29.67
Batch 280/400 | Loss: 0.0514 | PSNR: 24.02
Batch 300/400 | Loss: 0.0111 | PSNR: 37.25
Batch 320/400 | Loss: 0.0273 | PSNR: 28.62
Batch 340/400 | Loss: 0.0466 | PSNR: 24.86
Batch 360/400 | Loss: 0.0343 | PSNR: 28.98
Batch 380/400 | Loss: 0.0203 | PSNR: 32.99

Epoch Summary
Average Loss: 0.02699856203747913
Average PSNR: 30.08312931060791
New best model saved

Epoch 3/50


...

Batch 0/400 | Loss: 0.0302 | PSNR: 28.22
Batch 20/400 | Loss: 0.0154 | PSNR: 35.54
Batch 40/400 | Loss: 0.0246 | PSNR: 33.23
Batch 60/400 | Loss: 0.0228 | PSNR: 30.84
Batch 80/400 | Loss: 0.0171 | PSNR: 34.13
Batch 100/400 | Loss: 0.0241 | PSNR: 29.87
Batch 120/400 | Loss: 0.0251 | PSNR: 29.17
Batch 140/400 | Loss: 0.0103 | PSNR: 39.39
Batch 160/400 | Loss: 0.0296 | PSNR: 27.82
Batch 180/400 | Loss: 0.0240 | PSNR: 30.74
Batch 200/400 | Loss: 0.0478 | PSNR: 23.83
Batch 220/400 | Loss: 0.0356 | PSNR: 26.62
Batch 240/400 | Loss: 0.0052 | PSNR: 42.40
Batch 260/400 | Loss: 0.0306 | PSNR: 29.82
Batch 280/400 | Loss: 0.0273 | PSNR: 31.96
Batch 300/400 | Loss: 0.0164 | PSNR: 36.94
Batch 320/400 | Loss: 0.0186 | PSNR: 35.88
Batch 340/400 | Loss: 0.0170 | PSNR: 37.31
Batch 360/400 | Loss: 0.0139 | PSNR: 37.11
Batch 380/400 | Loss: 0.0207 | PSNR: 32.03

Epoch Summary
Average Loss: 0.022120301373070105
Average PSNR: 31.960313806533815

Epoch 49/50


/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:72: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  clean = imageio.imrea

Batch 0/400 | Loss: 0.0153 | PSNR: 36.08
Batch 20/400 | Loss: 0.0125 | PSNR: 36.10
Batch 40/400 | Loss: 0.0210 | PSNR: 31.16
Batch 60/400 | Loss: 0.0224 | PSNR: 30.09
Batch 80/400 | Loss: 0.0361 | PSNR: 26.07
Batch 100/400 | Loss: 0.0228 | PSNR: 30.36
Batch 120/400 | Loss: 0.0207 | PSNR: 31.64
Batch 140/400 | Loss: 0.0259 | PSNR: 28.55
Batch 160/400 | Loss: 0.0180 | PSNR: 32.50
Batch 180/400 | Loss: 0.0263 | PSNR: 28.93
Batch 200/400 | Loss: 0.0226 | PSNR: 30.13
Batch 220/400 | Loss: 0.0181 | PSNR: 37.17
Batch 240/400 | Loss: 0.0068 | PSNR: 39.54
Batch 260/400 | Loss: 0.0218 | PSNR: 30.81
Batch 280/400 | Loss: 0.0183 | PSNR: 33.91
Batch 300/400 | Loss: 0.0139 | PSNR: 33.02
Batch 320/400 | Loss: 0.0376 | PSNR: 27.02
Batch 340/400 | Loss: 0.0074 | PSNR: 40.65
Batch 360/400 | Loss: 0.0255 | PSNR: 29.69
Batch 380/400 | Loss: 0.0160 | PSNR: 33.34

Epoch Summary
Average Loss: 0.021877356842160224
Average PSNR: 31.963660488128664

Epoch 50/50


/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:71: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  noisy = imageio.imread(os.path.join(self.noisy_dir, name))
/tmp/ipykernel_55/3446730028.py:72: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  clean = imageio.imrea

Batch 0/400 | Loss: 0.0273 | PSNR: 28.99
Batch 20/400 | Loss: 0.0148 | PSNR: 32.69
Batch 40/400 | Loss: 0.0439 | PSNR: 24.79
Batch 60/400 | Loss: 0.0169 | PSNR: 30.98
Batch 80/400 | Loss: 0.0317 | PSNR: 27.57
Batch 100/400 | Loss: 0.0140 | PSNR: 36.34
Batch 120/400 | Loss: 0.0240 | PSNR: 30.09
Batch 140/400 | Loss: 0.0245 | PSNR: 35.25
Batch 160/400 | Loss: 0.0219 | PSNR: 30.28
Batch 180/400 | Loss: 0.0183 | PSNR: 31.28
Batch 200/400 | Loss: 0.0357 | PSNR: 26.98
Batch 220/400 | Loss: 0.0259 | PSNR: 29.03
Batch 240/400 | Loss: 0.0201 | PSNR: 31.17
Batch 260/400 | Loss: 0.0207 | PSNR: 32.41
Batch 280/400 | Loss: 0.0518 | PSNR: 23.19
Batch 300/400 | Loss: 0.0347 | PSNR: 26.74
Batch 320/400 | Loss: 0.0225 | PSNR: 30.92
Batch 340/400 | Loss: 0.0279 | PSNR: 32.92
Batch 360/400 | Loss: 0.0180 | PSNR: 35.57
Batch 380/400 | Loss: 0.0218 | PSNR: 30.18

Epoch Summary
Average Loss: 0.022209795473609118
Average PSNR: 31.754780321121217

Training complete
